# Train the Model on the Managed Cluster - Without Federated Learning

- **Current Cluster:** Cluster 1
- **Dataset (Subset):** MNIST (Digits 0, 1, 2, 3, 4)

In [ ]:
import os
os.environ["PATH"] += ":/home/jovyan/.local/bin"
# import sys
# sys.path.append('/home/jovyan/.local/bin')

!pip install -q flwr==1.13.1 'flwr-datasets[vision]>=0.4.0' torch==2.2.1 torchvision==0.17.1

In [ ]:
import numpy as np
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

In [ ]:
fds = None  # Cache FederatedDataset
def load_data(data_path: str):
    """Load partition MNIST data based on label ranges, partitioning only the training data."""
    # Only initialize `FederatedDataset` once
    global fds
    if fds is None:
        fds = FederatedDataset(
            dataset="mnist",
            partitioners={"train": 1},
        )

    # Load the dataset
    dataset = fds.load_partition(0, "train").with_format("numpy")

    X, y = dataset["image"].reshape((len(dataset), -1)), dataset["label"]

    # Define partition labels
    if "cluster1" in data_path:
        labels_to_include = [0, 1, 2, 3, 4]
    elif "cluster2" in data_path:
        labels_to_include = [5, 6, 7, 8, 9]
    else:
        raise ValueError(f"Invalid data_path {data_path}. Should be 'cluster1' or 'cluster2'.")

    # Filter the training data based on labels
    X_filtered = X[np.isin(y, labels_to_include)]
    y_filtered = y[np.isin(y, labels_to_include)]

    # Split the filtered data into 90% train and 10% test
    X_train, X_test = X_filtered[: int(0.9 * len(X_filtered))], X_filtered[int(0.1 * len(X_filtered)) :]
    y_train, y_test = y_filtered[: int(0.9 * len(y_filtered))], y_filtered[int(0.1 * len(y_filtered)) :]

    # Keep the test data intact (no partitioning)
    X_test_all = X[int(0.8 * len(X)) :]  # Full test data
    y_test_all = y[int(0.8 * len(y)) :]  # Full test data labels

    return (X_train, y_train), (X_test_all, y_test_all)

In [ ]:
# load the dataset from the managed cluster
train_data, test_data = load_data("/data/private/cluster1") 

# Extract features and labels
X_train, y_train = train_data
X_test, y_test = test_data
print(X_train.shape, y_train)
print(X_test.shape, y_test)

In [ ]:
def get_model(penalty: str, local_epochs: int):

    return LogisticRegression(
        penalty=penalty,
        max_iter=local_epochs,
        warm_start=True,
    )

model =  get_model("l2", 3) 

In [ ]:
# Train the model on the training data
model.fit(X_train, y_train)

In [ ]:
# Make predictions on the test data
y_pred = model.predict(X_test)

# Evaluate the model
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print("Model accuracy on the test data:", accuracy)

In [ ]:
# Plot predicted vs actual labels for some of the test samples
fig, axes = plt.subplots(1, 10, figsize=(15, 3))
start = 10
end = 20
for i in range(start,end):
    ax = axes[i-start]
    ax.imshow(X_test[i].reshape(28, 28), cmap="gray")

    color = "green" if y_pred[i] == y_test[i] else "red"
    ax.set_title(f"Pred: {y_pred[i]}", color=color)
    ax.axis("off")
plt.tight_layout()
plt.show()